# 03. Model and Cost Comparison

**Purpose:** Compare practical trade-offs between models (cost/speed vs accuracy) by running the same prompt on two models.

**Models:**
- Baseline: **gemini-2.5-flash**
- Advanced: **gemini-2.5-pro**

**Key Takeaway:** Model selection is a business decision balancing accuracy, latency, and cost.


## Step 0: Set up environment & authenticate

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git

from pathlib import Path
import sys
PROJECT_ROOT = [it for it in [Path("src"), Path('esmt-workshop/src')] if (it / 'esmt_workshop').exists()][0].parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)
%pip install -q -r {PROJECT_ROOT}/requirements.txt

import pandas as pd
import os
from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report, publish_to_leaderboard
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
from esmt_workshop.authenticate import authenticate

STUDENT_EMAIL = authenticate()
from esmt_workshop.preset_params import get_preset_params
preset_params = get_preset_params()

## Step 1: Limited-Run Model Comparison (edit **one cell only**)

Use the next cell to compare models quickly on a small subset.

**Do this in order:**
1. Paste your best prompt from Notebook 02 into `STUDENT_PROMPT_TEMPLATE`.
2. **Run the baseline model first**:
   - Set `MODEL_NAME = FLASH_MODEL`
   - Set `STAGE_NAME = 'baseline'`
   - Run the cell → record `micro_accuracy` + `Runtime (sec)`.
3. **Run the advanced model second**:
   - Set `MODEL_NAME = PRO_MODEL`
   - Set `STAGE_NAME = 'advanced'`
   - Run the same cell again → record `micro_accuracy` + `Runtime (sec)`.

**Rule:** Only edit:
- `STUDENT_PROMPT_TEMPLATE` / `PROMPT_TO_RUN`
- `MODEL_NAME`
- `STAGE_NAME`


In [ ]:
import time

# -----------------------------
# PROMPT (paste your best prompt from Notebook 02)
# -----------------------------
STUDENT_PROMPT_TEMPLATE = '''You are a highly precise AML address parsing system.

Input address:
{address}

Return STRICT JSON only using this exact schema:
{schema}

Rules:
- Town Name: city/locality only (no street/state/province).
- Postal Code: postal token(s) only (preserve formatting).
- Remaining Address: everything else (street, number, unit, state/province, etc.).
- Country Code: ISO alpha-2 uppercase (convert country names when explicit).
- Do not invent values; use "" when unknown.
'''

PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE


# -----------------------------
# MODEL + STAGE (students change these)
# -----------------------------
FLASH_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
PRO_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

# Run baseline first:
MODEL_NAME = FLASH_MODEL
STAGE_NAME = 'baseline'   # must be one of: baseline, prompt_tuned, advanced, two_stage

# Then switch to:
# MODEL_NAME = PRO_MODEL
# STAGE_NAME = 'advanced'


# -----------------------------
# FIXED PARAMS (from original notebooks)
# -----------------------------
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
USE_GUARDRAILS = False  # default for this notebook

# Execution params (don't change)
MAX_TOKENS = preset_params["MAX_TOKENS"]
MAX_WORKERS = preset_params["MAX_WORKERS"]


# -----------------------------
# DATA (limited run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
dev_small = dev_df.head(10).copy()


# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_small,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Generate report (same output style as original notebooks)
report = evaluate_predictions(pred_df, dev_small)

print(report['summary'])
display(report['field_metrics'])


## Step 2: Full Dataset Run (your **publishable** result)

Choose **one** model to run on the full dev set. This score is what you will publish later.

**Pick ONE configuration and run this cell once:**

- **Baseline publish run**:
  - `MODEL_NAME = FLASH_MODEL`
  - `STAGE_NAME = 'baseline'`

- **Advanced publish run**:
  - `MODEL_NAME = PRO_MODEL`
  - `STAGE_NAME = 'advanced'`

**Rule:** Only edit:
- `STUDENT_PROMPT_TEMPLATE` / `PROMPT_TO_RUN`
- `MODEL_NAME`
- `STAGE_NAME`


In [ ]:
import time

# -----------------------------
# PROMPT (paste the SAME best prompt you used above)
# -----------------------------
STUDENT_PROMPT_TEMPLATE = '''You are a highly precise AML address parsing system.

Input address:
{address}

Return STRICT JSON only using this exact schema:
{schema}

Rules:
- Town Name: city/locality only (no street/state/province).
- Postal Code: postal token(s) only (preserve formatting).
- Remaining Address: everything else (street, number, unit, state/province, etc.).
- Country Code: ISO alpha-2 uppercase (convert country names when explicit).
- Do not invent values; use "" when unknown.
'''

PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE


# -----------------------------
# MODEL + STAGE (students choose ONE for publishable result)
# -----------------------------
FLASH_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
PRO_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

# Choose ONE:
MODEL_NAME = FLASH_MODEL
STAGE_NAME = 'baseline'

# Or:
# MODEL_NAME = PRO_MODEL
# STAGE_NAME = 'advanced'


# -----------------------------
# FIXED PARAMS (from original notebooks)
# -----------------------------
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
USE_GUARDRAILS = False


# -----------------------------
# DATA (full run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')


# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

report = evaluate_predictions(pred_df, dev_df)

print(report['summary'])
display(report['field_metrics'])


## Step 3: Publish Placeholder

Later, you will publish the results from **Step 6**. For now, this is a placeholder.

- Save predictions/report artifacts
- Submit score via the workshop publishing mechanism (to be provided)


In [ ]:
publish_to_leaderboard(report, STUDENT_EMAIL)